In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ludhiana_rainfall_target.csv"
)

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Features available from the current day's weather.
# Exclude date, target, and calendar columns for now.
feature_columns = [
    "temperature_mean",
    "temperature_max",
    "temperature_min",
    "humidity_mean",
    "dew_point_mean",
    "apparent_temperature_max",
    "precipitation_sum",
    "rain_sum",
    "pressure_mean",
    "cloud_cover_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_gusts_max",
    "month",
    "day_of_week",
    "day_of_year"
]

target_column = "tomorrow_rainfall_mm"

# Train on the earlier 80%; test on the later 20%.
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("DATA PREPARATION SUCCESSFUL")
print("-" * 40)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training dates:", train_df["date"].min().date(),
      "to", train_df["date"].max().date())
print("Testing dates:", test_df["date"].min().date(),
      "to", test_df["date"].max().date())
print("Number of features:", len(feature_columns))
print("Missing values in training features:", X_train.isna().sum().sum())
print("Missing values in testing features:", X_test.isna().sum().sum())
print("Target:", target_column)

print("\nTest target summary:")
print(y_test.describe())


In [ ]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

PROJECT_DIR = Path.cwd().parent
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Train the first Random Forest model
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict rainfall for the unseen test period
predictions = model.predict(X_test)

# Rainfall cannot be negative
predictions = np.maximum(predictions, 0)

# Evaluate model performance
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("RANDOM FOREST — FIRST RESULTS")
print("-" * 40)
print(f"MAE:  {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")
print(f"R²:   {r2:.3f}")

# Save the trained model and its feature names
joblib.dump(
    {
        "model": model,
        "feature_columns": feature_columns,
        "target_column": target_column
    },
    MODEL_DIR / "ludhiana_rainfall_random_forest.joblib"
)

# Save test predictions for later comparison
results = pd.DataFrame({
    "date": test_df["date"].values,
    "actual_rainfall_mm": y_test.values,
    "predicted_rainfall_mm": predictions
})

results.to_csv(
    REPORT_DIR / "random_forest_test_predictions.csv",
    index=False
)

print("\nModel saved to:", MODEL_DIR / "ludhiana_rainfall_random_forest.joblib")
print("Test predictions saved to:", REPORT_DIR / "random_forest_test_predictions.csv")

print("\nFirst 10 predictions:")
display(results.head(10).round(2))


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Baseline: always predict the average rainfall from training data
baseline_prediction = np.full(len(y_test), y_train.mean())

def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "MAE (mm)": mean_absolute_error(actual, predicted),
        "RMSE (mm)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    }

comparison = pd.DataFrame([
    evaluate_model("Training-mean baseline", y_test, baseline_prediction),
    evaluate_model("Random Forest", y_test, predictions)
])

print("MODEL COMPARISON")
display(comparison.round(3))

# Evaluate separately on rainy test days
rainy_mask = y_test.to_numpy() > 0

print("\nRAINY-DAY PERFORMANCE")
print("Rainy test days:", int(rainy_mask.sum()))

if rainy_mask.sum() > 0:
    actual_rainy = y_test.to_numpy()[rainy_mask]
    predicted_rainy = predictions[rainy_mask]

    print(
        "Rainy-day MAE:",
        round(mean_absolute_error(actual_rainy, predicted_rainy), 3),
        "mm"
    )
    print(
        "Rainy-day RMSE:",
        round(np.sqrt(mean_squared_error(actual_rainy, predicted_rainy)), 3),
        "mm"
    )

# Check how often the model predicts rain
print("\nRAIN DETECTION CHECK")
print("Actual rainy days:", int((y_test > 0).sum()))
print("Predictions above 0.1 mm:", int((predictions > 0.1).sum()))
print("Predictions above 1 mm:", int((predictions >= 1).sum()))


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importance from the trained Random Forest
importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("RANDOM FOREST FEATURE IMPORTANCE")
display(importance_df.round(4))

# Visualize the 12 most influential features
top_features = importance_df.head(12).sort_values("Importance")

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("RainfallAI — Top 12 Feature Importances")
plt.xlabel("Relative importance")
plt.ylabel("Weather feature")
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load the historical daily forecast data
forecast_path = RAW_DIR / "ludhiana_historical_daily_forecast.csv"
forecast_df = pd.read_csv(forecast_path, parse_dates=["date"])
forecast_df = forecast_df.sort_values("date").reset_index(drop=True)

# Create tomorrow's rainfall target from the following calendar day
forecast_df["tomorrow_rainfall_mm"] = forecast_df["rain_sum"].shift(-1)

# Drop the final date because the next day's target is unavailable
forecast_model_df = forecast_df.dropna(
    subset=["tomorrow_rainfall_mm"]
).copy()

# Save a separate dataset for the new model
output_path = PROCESSED_DIR / "ludhiana_forecast_model_data.csv"
forecast_model_df.to_csv(output_path, index=False)

print("FORECAST MODEL DATA PREPARED")
print("-" * 40)
print("Rows:", len(forecast_model_df))
print("Columns:", len(forecast_model_df.columns))
print("Date range:", forecast_model_df["date"].min().date(),
      "to", forecast_model_df["date"].max().date())
print("Missing values:", forecast_model_df.isna().sum().sum())

print("\nCheck target alignment:")
display(
    forecast_model_df[
        ["date", "rain_sum", "tomorrow_rainfall_mm"]
    ].head(7)
)

print("\nSaved to:", output_path)


In [ ]:

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

data_path = PROCESSED_DIR / "ludhiana_forecast_model_data.csv"
df_forecast = pd.read_csv(data_path, parse_dates=["date"])
df_forecast = df_forecast.sort_values("date").reset_index(drop=True)

# Exclude rainfall totals from the current day to avoid a shortcut.
feature_columns_forecast = [
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "dew_point_2m_mean",
    "cloud_cover_mean",
    "wind_speed_10m_max"
]

target_column_forecast = "tomorrow_rainfall_mm"

split_index = int(len(df_forecast) * 0.80)
train_forecast = df_forecast.iloc[:split_index].copy()
test_forecast = df_forecast.iloc[split_index:].copy()

X_train_forecast = train_forecast[feature_columns_forecast]
y_train_forecast = train_forecast[target_column_forecast]
X_test_forecast = test_forecast[feature_columns_forecast]
y_test_forecast = test_forecast[target_column_forecast]

model_forecast = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model_forecast.fit(X_train_forecast, y_train_forecast)
pred_forecast = np.maximum(model_forecast.predict(X_test_forecast), 0)

mae = mean_absolute_error(y_test_forecast, pred_forecast)
rmse = np.sqrt(mean_squared_error(y_test_forecast, pred_forecast))
r2 = r2_score(y_test_forecast, pred_forecast)

print("FORECAST-FEATURE MODEL RESULTS")
print("-" * 40)
print("Training rows:", len(X_train_forecast))
print("Testing rows:", len(X_test_forecast))
print("Testing dates:", test_forecast["date"].min().date(),
      "to", test_forecast["date"].max().date())
print("Features:", feature_columns_forecast)
print(f"MAE:  {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")
print(f"R²:   {r2:.3f}")

joblib.dump(
    {
        "model": model_forecast,
        "feature_columns": feature_columns_forecast,
        "target_column": target_column_forecast
    },
    MODEL_DIR / "ludhiana_forecast_feature_model.joblib"
)

results_forecast = pd.DataFrame({
    "date": test_forecast["date"].values,
    "actual_rainfall_mm": y_test_forecast.values,
    "predicted_rainfall_mm": pred_forecast
})

results_forecast.to_csv(
    REPORT_DIR / "forecast_feature_test_predictions.csv",
    index=False
)

print("\nFirst 10 test predictions:")
display(results_forecast.head(10).round(2))
print("\nModel and predictions saved successfully.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path.cwd().parent
REPORT_DIR = PROJECT_DIR / "reports"

model1_path = REPORT_DIR / "random_forest_test_predictions.csv"
model2_path = REPORT_DIR / "forecast_feature_test_predictions.csv"

model1 = pd.read_csv(model1_path, parse_dates=["date"])
model2 = pd.read_csv(model2_path, parse_dates=["date"])

# Rename prediction columns to identify each model
model1 = model1.rename(
    columns={"predicted_rainfall_mm": "model1_prediction_mm"}
)
model2 = model2.rename(
    columns={
        "actual_rainfall_mm": "actual_model2_mm",
        "predicted_rainfall_mm": "model2_prediction_mm"
    }
)

# Align both models by date
comparison_df = model1.merge(model2, on="date", how="inner")

# Verify actual targets match
assert np.allclose(
    comparison_df["actual_rainfall_mm"],
    comparison_df["actual_model2_mm"]
), "The actual target values do not match!"

actual = comparison_df["actual_rainfall_mm"]

def metrics(name, predicted):
    return {
        "Model": name,
        "MAE (mm)": mean_absolute_error(actual, predicted),
        "RMSE (mm)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    }

comparison_results = pd.DataFrame([
    metrics("Model 1 — All features", comparison_df["model1_prediction_mm"]),
    metrics("Model 2 — Forecast features", comparison_df["model2_prediction_mm"])
])

print("COMMON TEST PERIOD")
print("Rows:", len(comparison_df))
print("Dates:", comparison_df["date"].min().date(), "to",
      comparison_df["date"].max().date())

print("\nMODEL COMPARISON")
display(comparison_results.round(3))

print("\nALIGNED PREDICTIONS — FIRST 10 DAYS")
display(
    comparison_df[
        [
            "date",
            "actual_rainfall_mm",
            "model1_prediction_mm",
            "model2_prediction_mm"
        ]
    ].head(10).round(2)
)


In [ ]:

print("Comparison rows:", len(comparison_df))

print("\nMissing values:")
display(comparison_df.isna().sum())

print("\nLast 10 aligned rows:")
display(
    comparison_df[
        [
            "date",
            "actual_rainfall_mm",
            "model1_prediction_mm",
            "model2_prediction_mm"
        ]
    ].tail(10)
)

print("\nPrediction ranges:")
print(
    comparison_df[
        ["model1_prediction_mm", "model2_prediction_mm"]
    ].describe().round(2)
)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

data_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ludhiana_forecast_model_data.csv"
)

forecast_data = pd.read_csv(data_path, parse_dates=["date"])

print("Dataset shape:", forecast_data.shape)
print("\nDate range:")
print(forecast_data["date"].min(), "to", forecast_data["date"].max())

print("\nColumns:")
print(forecast_data.columns.tolist())

print("\nFirst 5 records:")
display(forecast_data.head())

print("\nLast 5 records:")
display(forecast_data.tail())

print("\nMissing values:")
display(forecast_data.isna().sum())


In [ ]:

import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path.cwd().parent
DATA_PATH = PROJECT_DIR / "data" / "processed" / "ludhiana_2025_ml_training_data.csv"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

features = [
    "temperature_mean_c",
    "humidity_mean_pct",
    "dew_point_mean_c",
    "cloud_cover_mean_pct",
    "wind_speed_mean_kmh"
]
target = "observed_rainfall_mm"

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
predictions = np.maximum(model.predict(X_test), 0)

# Baseline: predict the average rainfall from the training period.
baseline_predictions = np.full(len(y_test), y_train.mean())

def calculate_metrics(name, actual, predicted):
    return {
        "Model": name,
        "MAE (mm)": mean_absolute_error(actual, predicted),
        "RMSE (mm)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted)
    }

results = pd.DataFrame([
    calculate_metrics("Random Forest", y_test, predictions),
    calculate_metrics("Training-mean baseline", y_test, baseline_predictions)
])

print("TRAINING PERIOD:")
print(train_df["date"].min().date(), "to", train_df["date"].max().date())

print("\nTEST PERIOD:")
print(test_df["date"].min().date(), "to", test_df["date"].max().date())

print("\nMODEL EVALUATION:")
display(results.round(3))

test_results = pd.DataFrame({
    "date": test_df["date"],
    "actual_rainfall_mm": y_test.to_numpy(),
    "predicted_rainfall_mm": predictions
})

test_results.to_csv(REPORT_DIR / "ludhiana_2025_test_predictions.csv", index=False)
joblib.dump(model, MODEL_DIR / "ludhiana_2025_rainfall_model.joblib")

print("\nSaved model:", MODEL_DIR / "ludhiana_2025_rainfall_model.joblib")
print("Saved predictions:", REPORT_DIR / "ludhiana_2025_test_predictions.csv")


In [ ]:

# STEP 61 — Load data for model training

from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path("../data/raw/ludhiana_rainfall_reference_2022_2025.csv")

if not data_path.exists():
    data_path = Path("data/raw/ludhiana_rainfall_reference_2022_2025.csv")

training_df = pd.read_csv(data_path)
training_df["date"] = pd.to_datetime(training_df["date"])
training_df = training_df.sort_values("date").reset_index(drop=True)

print("TRAINING DATA LOADED")
print("-" * 35)
print("Dataset shape:", training_df.shape)
print("First date:", training_df["date"].min().date())
print("Last date:", training_df["date"].max().date())
print("Missing values:", training_df.isna().sum().sum())
print("\nFirst 5 rows:")
print(training_df.head().to_string(index=False))


In [ ]:

# STEP 62 — Create prediction target

training_df["tomorrow_rainfall_mm"] = (
    training_df["observed_rainfall_mm"].shift(-1)
)

training_df["month"] = training_df["date"].dt.month
training_df["day_of_year"] = training_df["date"].dt.dayofyear
training_df["day_of_week"] = training_df["date"].dt.dayofweek

training_df["season"] = training_df["month"].apply(
    lambda m: (
        "Winter" if m in [12, 1, 2] else
        "Summer" if m in [3, 4, 5] else
        "Monsoon" if m in [6, 7, 8, 9] else
        "Post-Monsoon"
    )
)

training_df = training_df.dropna(
    subset=["tomorrow_rainfall_mm"]
).reset_index(drop=True)

print("Rows for training:", len(training_df))
print(training_df[[
    "date", "observed_rainfall_mm", "tomorrow_rainfall_mm",
    "month", "season"
]].head())


In [ ]:

# STEP 63 — Select features and target

feature_columns = [
    "observed_rainfall_mm",
    "month",
    "day_of_year",
    "day_of_week",
    "season"
]

X = training_df[feature_columns].copy()
y = training_df["tomorrow_rainfall_mm"].copy()

print("Feature columns:", feature_columns)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget summary:")
print(y.describe())


In [ ]:

# STEP 64 — Encode categorical features

X_encoded = pd.get_dummies(
    X,
    columns=["season"],
    dtype=int
)

print("Encoded feature columns:")
print(X_encoded.columns.tolist())
print("\nEncoded shape:", X_encoded.shape)
print(X_encoded.head())


In [ ]:

# STEP 65 — Chronological train/test split

split_index = int(len(training_df) * 0.80)

X_train = X_encoded.iloc[:split_index].copy()
X_test = X_encoded.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training dates:",
      training_df["date"].iloc[0].date(), "to",
      training_df["date"].iloc[split_index - 1].date())
print("Testing dates:",
      training_df["date"].iloc[split_index].date(), "to",
      training_df["date"].iloc[-1].date())


In [ ]:

# STEP 66 — Baseline persistence prediction

baseline_predictions = X_test["observed_rainfall_mm"].to_numpy()
baseline_mae = np.mean(np.abs(y_test.to_numpy() - baseline_predictions))
baseline_rmse = np.sqrt(
    np.mean((y_test.to_numpy() - baseline_predictions) ** 2)
)

print("BASELINE RESULTS")
print("MAE:", round(baseline_mae, 3), "mm")
print("RMSE:", round(baseline_rmse, 3), "mm")


In [ ]:

# STEP 67 — Train Random Forest model

from sklearn.ensemble import RandomForestRegressor

random_forest = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)

print("Random Forest training completed.")
print("Sample predictions:", np.round(rf_predictions[:10], 2))


In [ ]:

# STEP 68 — Train Extra Trees model

from sklearn.ensemble import ExtraTreesRegressor

extra_trees = ExtraTreesRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

extra_trees.fit(X_train, y_train)

et_predictions = extra_trees.predict(X_test)

print("Extra Trees training completed.")
print("Sample predictions:", np.round(et_predictions[:10], 2))


In [ ]:

# STEP 69 — Train Gradient Boosting model

from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    loss="huber",
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

gb_predictions = gradient_boosting.predict(X_test)

print("Gradient Boosting training completed.")
print("Sample predictions:", np.round(gb_predictions[:10], 2))


In [ ]:

# STEP 70 — Compare model performance

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

predictions_by_model = {
    "Persistence Baseline": baseline_predictions,
    "Random Forest": rf_predictions,
    "Extra Trees": et_predictions,
    "Gradient Boosting": gb_predictions
}

results = []

for model_name, predictions in predictions_by_model.items():
    results.append({
        "Model": model_name,
        "MAE_mm": mean_absolute_error(y_test, predictions),
        "RMSE_mm": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("MAE_mm")

print(results_df.round(3).to_string(index=False))


In [ ]:

# STEP 71 — Compare MAE visually

plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["MAE_mm"])

plt.title("Model Comparison — Mean Absolute Error")
plt.xlabel("Model")
plt.ylabel("MAE (mm; lower is better)")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 72 — Select the best model

best_model_name = results_df.iloc[0]["Model"]

model_objects = {
    "Random Forest": random_forest,
    "Extra Trees": extra_trees,
    "Gradient Boosting": gradient_boosting
}

if best_model_name == "Persistence Baseline":
    best_model = None
    print("The baseline performed best. Do not deploy a weaker ML model.")
else:
    best_model = model_objects[best_model_name]
    print("Best model by test MAE:", best_model_name)

print("Best measured MAE:",
      round(results_df.iloc[0]["MAE_mm"], 3), "mm")


In [ ]:

# STEP 73 — Feature importance

if best_model is not None and hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=False)

    print(importance_df.round(4).to_string(index=False))

    plt.figure(figsize=(9, 5))
    plt.barh(
        importance_df["Feature"][::-1],
        importance_df["Importance"][::-1]
    )
    plt.title(f"Feature Importance — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance unavailable for the selected model.")


In [ ]:

# STEP 74 — Save model artifacts

import joblib

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

model_dir = project_root / "model"
model_dir.mkdir(parents=True, exist_ok=True)

if best_model is not None:
    joblib.dump(best_model, model_dir / "rainfall_model.joblib")

joblib.dump(
    {
        "feature_columns": feature_columns,
        "encoded_feature_columns": X_encoded.columns.tolist(),
        "target_column": "tomorrow_rainfall_mm",
        "rain_threshold_mm": 0.1,
        "best_model_name": best_model_name
    },
    model_dir / "feature_config.joblib"
)

print("Model folder:", model_dir.resolve())
print("Selected model:", best_model_name)
print("Configuration saved.")


In [ ]:

# STEP 75 — Save training results

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    processed_dir / "model_training_comparison.csv",
    index=False
)

print("Training phase completed.")
print("Results saved to:",
      (processed_dir / "model_training_comparison.csv").resolve())
print("\nFinal comparison:")
print(results_df.round(3).to_string(index=False))


In [ ]:

# STEP 13 — Inspect rainfall distribution

print("TRAINING DATA")
print("-" * 40)
print("Training rows:", len(train_df))
print("Training dates:", train_df["date"].min(), "to", train_df["date"].max())
print("Rainfall total:", round(y_train.sum(), 2), "mm")
print("Average rainfall:", round(y_train.mean(), 3), "mm")
print("Maximum rainfall:", round(y_train.max(), 2), "mm")
print("Days with rainfall >= 1 mm:", int((y_train >= 1).sum()))
print("Days with zero rainfall:", int((y_train == 0).sum()))

print("\nTESTING DATA")
print("-" * 40)
print("Testing rows:", len(test_df))
print("Testing dates:", test_df["date"].min(), "to", test_df["date"].max())
print("Rainfall total:", round(y_test.sum(), 2), "mm")
print("Average rainfall:", round(y_test.mean(), 3), "mm")
print("Maximum rainfall:", round(y_test.max(), 2), "mm")
print("Days with rainfall >= 1 mm:", int((y_test >= 1).sum()))
print("Days with zero rainfall:", int((y_test == 0).sum()))


In [ ]:

print("Data consistency check")
print("-" * 40)

print("df shape:", df.shape)
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

print("\nTarget lengths:")
print("y_train:", len(y_train))
print("y_test:", len(y_test))

print("\nTarget counts:")
print("Training target values:")
print(y_train.value_counts().head(10))

print("\nTesting target values:")
print(y_test.value_counts().head(10))

print("\nTarget summary:")
print("Training:", y_train.describe())
print("\nTesting:", y_test.describe())


In [ ]:

# STEP 15 — Rebuild the train/test variables consistently

df = df.sort_values("date").reset_index(drop=True)

feature_columns = [
    "temperature_mean",
    "temperature_max",
    "temperature_min",
    "humidity_mean",
    "dew_point_mean",
    "apparent_temperature_max",
    "precipitation_sum",
    "rain_sum",
    "pressure_mean",
    "cloud_cover_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_gusts_max",
    "month",
    "day_of_week",
    "day_of_year"
]

target_column = "tomorrow_rainfall_mm"

# Confirm every required column exists
required_columns = feature_columns + [target_column]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in df: {missing_columns}")

# Use only the dataset currently loaded in df
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Training rows:", len(X_train))
print("Training target values:", len(y_train))
print("Testing rows:", len(X_test))
print("Testing target values:", len(y_test))

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

print("\nVariables rebuilt successfully!")


In [ ]:

# STEP 16 — Identify and load the correct training dataset

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

data_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ludhiana_rainfall_target.csv"
)

df_model = pd.read_csv(data_path, parse_dates=["date"])
df_model = df_model.sort_values("date").reset_index(drop=True)

print("Dataset loaded:", data_path.name)
print("Shape:", df_model.shape)
print("\nAvailable columns:")
print(df_model.columns.tolist())

print("\nDate range:")
print(df_model["date"].min(), "to", df_model["date"].max())

print("\nFirst five rows:")
display(df_model.head())
